# Week 3 Lab — Control flow and functions

**HWRS 564a · Fall 2026**

Last week you wrote single expressions. This week you write code that *decides*
and code that *repeats*, and you package it up so you can use it twice.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked  **`# YOUR TURN`**  have
something for you to write. Cells marked **`# CHECK`** verify your answer — if
they run without complaint, you're right.

> **Before you submit anything all semester:** *Kernel → Restart Kernel and Run
> All Cells*. A notebook that only works when run out of order is not finished.


## Learning objectives

By the end of this notebook you can:

1. Branch on a condition with `if` / `elif` / `else`
2. Repeat work with a `for` loop, and accumulate a result as you go
3. Loop until a condition is met with `while`, without hanging your kernel
4. Write a function with arguments, a default, a docstring, and a return value
5. Guard a function against input that doesn't make sense
6. Build a list from another list with a comprehension

---

## Part 1 — Making decisions

A monitoring well is *dry* if the water level has dropped below the bottom of
its screen. Deciding that in code looks like this.

In [ ]:
water_level = 88.4      # m below land surface
screen_bottom = 92.0    # m below land surface

if water_level > screen_bottom:
    print("Dry — the water table is below the screen.")
else:
    print("Wet — the well still has water in it.")

Two things to notice, because both bite people in week 3:

- The `:` at the end of the `if` line is required.
- **The indented block is the body.** Python uses indentation the way other
  languages use braces. Four spaces, consistently.

For more than two outcomes, chain with `elif`. Python takes the **first** branch
that is true and skips the rest.

In [ ]:
decline_m = 14.2   # how far the water table has dropped over the record

if decline_m > 30:
    status = "severe"
elif decline_m > 10:
    status = "moderate"
elif decline_m > 0:
    status = "mild"
else:
    status = "stable or recovering"

print(f"A {decline_m} m decline is {status}.")

Order matters. If the `> 0` test came first, *every* declining well would be
called mild and the other branches would be unreachable.

### YOUR TURN 1

An unconfined aquifer is being pumped. Write the branching that sets
`recommendation` based on the **percentage of saturated thickness** already
dewatered:

| Dewatered | `recommendation` |
|---|---|
| more than 50% | `"stop pumping"` |
| more than 25% | `"reduce pumping"` |
| anything else | `"continue monitoring"` |

In [ ]:
saturated_thickness_m = 62.0
drawdown_m = 19.5

percent_dewatered = 100 * drawdown_m / saturated_thickness_m

# YOUR TURN
recommendation = ...

In [ ]:
# CHECK
assert isinstance(recommendation, str), "recommendation should be a string"
assert recommendation == "reduce pumping", f"got {recommendation!r}"
print(f"{percent_dewatered:.1f}% dewatered -> {recommendation}. Correct.")

# And the branches you didn't hit, checked by hand:
for dd, expected in [(35.0, "stop pumping"), (5.0, "continue monitoring")]:
    pct = 100 * dd / saturated_thickness_m
    print(f"  a {dd} m drawdown is {pct:.1f}% — should be {expected!r}")

---

## Part 2 — Repeating yourself, properly

A `for` loop walks through a sequence, one item at a time.

In [ ]:
depths_to_water = [31.2, 45.8, 88.4, 12.0, 67.3]   # m below land surface

for depth in depths_to_water:
    print(f"water at {depth} m")

The usual reason to loop is to **accumulate** something. Start with an empty
container, add to it inside the loop, use it after.

In [ ]:
total = 0.0
for depth in depths_to_water:
    total = total + depth

print(f"sum:  {total}")
print(f"mean: {total / len(depths_to_water):.2f} m")

Building a *list* instead of a number follows the same shape:

In [ ]:
FEET_PER_METRE = 3.28084

depths_ft = []
for depth in depths_to_water:
    depths_ft.append(depth * FEET_PER_METRE)

print(depths_ft)

When you need the position as well as the value, use `enumerate`. When you need
to walk two sequences together, use `zip`. Reach for these before you reach for
`range(len(...))`.

In [ ]:
well_ids = ["A-14", "B-07", "C-22", "D-03", "E-11"]

for i, (well, depth) in enumerate(zip(well_ids, depths_to_water)):
    print(f"{i}: {well:5s} {depth:6.1f} m")

### YOUR TURN 2

A well goes dry when the water level falls below its screen bottom. Using the
two lists below, build `dry_wells` — a list of the **IDs** of wells that are
dry — and count them in `n_dry`.

Use a `for` loop with `zip`, an `if` inside it, and `.append()`.

In [ ]:
screen_bottoms = [40.0, 52.0, 80.0, 35.0, 70.0]   # m below land surface, same order

# YOUR TURN
dry_wells = ...
n_dry = ...

In [ ]:
# CHECK
assert isinstance(dry_wells, list), "dry_wells should be a list"
assert dry_wells == ["C-22"], f"got {dry_wells}"
assert n_dry == 1, f"expected 1, got {n_dry}"
print(f"{n_dry} dry well: {dry_wells}. Correct.")

**Worth noticing:** C-22 is dry because 88.4 m is *below* 80.0 m. Depth below
land surface counts downward, so bigger means deeper. Half the sign errors in
groundwater work come from mixing depth-below-surface with elevation.

---

## Part 3 — `while`, and how not to hang your kernel

A `for` loop runs a known number of times. A `while` loop runs until a condition
stops being true — which is what you want when you don't know how many steps it
takes.

In [ ]:
storage = 100.0      # units of water in the reservoir
k = 0.15             # fraction lost each day
day = 0

while storage > 50.0:
    storage = storage * (1 - k)
    day = day + 1

print(f"storage fell below half after {day} days ({storage:.1f} left)")

> **The one rule:** something inside the loop must eventually make the condition
> false. If `storage` never shrank, that cell would run forever and you'd have to
> interrupt the kernel (the ■ button, or *Kernel → Interrupt*).

Because it is easy to get that wrong, real iterative code carries a **maximum
iteration count** as a safety net. You will see this exact pattern again in
Week 4 when you write a solver, and in Week 12 when MODFLOW refuses to converge.

In [ ]:
storage = 100.0
day = 0
MAX_DAYS = 1000

while storage > 50.0 and day < MAX_DAYS:
    storage = storage * (1 - k)
    day += 1          # `+=` is shorthand for `day = day + 1`

if day == MAX_DAYS:
    print("gave up — did not converge")
else:
    print(f"converged in {day} days")

---

## Part 4 — Functions

A function names a piece of work so you can do it again without copying it.

In [ ]:
def specific_discharge(hydraulic_conductivity, gradient):
    """Darcy specific discharge q = K * i.

    Parameters
    ----------
    hydraulic_conductivity : float
        K, in m/d.
    gradient : float
        Dimensionless hydraulic gradient, dh/dl.

    Returns
    -------
    float
        Specific discharge in m/d.
    """
    return hydraulic_conductivity * gradient


print(specific_discharge(12.0, 0.005059))

The anatomy:

- `def name(arguments):` — the header
- the **docstring** in triple quotes, saying what it does and what the units are
- `return` — what the caller gets back

A function with no `return` gives you `None`. That is a real and common bug:
`print` shows a value, `return` hands it back.

In [ ]:
def no_return(x):
    print(x * 2)          # prints, but hands back nothing

result = no_return(21)
print(f"result is {result!r} — see the problem?")

Arguments can have **defaults**, which makes the common case short and the
unusual case still possible.

In [ ]:
def volumetric_flow(hydraulic_conductivity, gradient, area, porosity=1.0):
    """Volumetric flow Q = q * A. Pass porosity to get seepage velocity instead."""
    return hydraulic_conductivity * gradient * area / porosity


print(f"{volumetric_flow(12.0, 0.005059, 45000):.1f} m3/d")
print(f"{volumetric_flow(12.0, 0.005059, 45000, porosity=0.28):.1f} m3/d of pore flow")

### YOUR TURN 3

Write `transmissivity(hydraulic_conductivity, thickness)`, which returns
$T = Kb$ in m²/d.

Give it a docstring. It should have **no** default arguments.

In [ ]:
# YOUR TURN
def transmissivity(...):
    ...

In [ ]:
# CHECK
assert callable(transmissivity), "transmissivity should be a function"
assert transmissivity.__doc__, "give it a docstring — you will thank yourself in week 12"
assert abs(transmissivity(12.0, 25.0) - 300.0) < 1e-9, "T = K * b"
assert abs(transmissivity(0.4, 110.0) - 44.0) < 1e-9, "check it works for other numbers too"
print(f"T = {transmissivity(12.0, 25.0)} m2/d for K = 12 m/d over 25 m. Correct.")

### YOUR TURN 4 — refuse bad input

A negative thickness is not a thin aquifer; it is a mistake upstream in someone's
code. A function that silently returns a negative transmissivity lets that
mistake travel.

Write `safe_transmissivity(hydraulic_conductivity, thickness)` that behaves like
`transmissivity` but **raises `ValueError`** if either argument is negative.

```python
raise ValueError("thickness must be positive, got -5")
```

In [ ]:
# YOUR TURN
def safe_transmissivity(...):
    ...

In [ ]:
# CHECK
assert abs(safe_transmissivity(12.0, 25.0) - 300.0) < 1e-9, "the normal case still has to work"

for bad_args in [(12.0, -25.0), (-12.0, 25.0)]:
    try:
        safe_transmissivity(*bad_args)
    except ValueError:
        print(f"  {bad_args} correctly raised ValueError")
    else:
        raise AssertionError(f"{bad_args} should have raised ValueError")

print("Correct.")

This is the habit worth forming: **fail loudly, near the mistake.** A model that
stops with a clear message costs you two minutes. A model that quietly produces
a plausible wrong answer costs you a week.

---

## Part 5 — Comprehensions

Building a list by looping and appending is so common that Python has a shorthand
for it. These two cells do exactly the same thing.

In [ ]:
# the long way
depths_ft = []
for depth in depths_to_water:
    depths_ft.append(depth * FEET_PER_METRE)

# the comprehension
depths_ft_2 = [depth * FEET_PER_METRE for depth in depths_to_water]

print(depths_ft == depths_ft_2)

A comprehension can filter, too — the `if` goes at the end:

In [ ]:
deep = [d for d in depths_to_water if d > 40]
print(deep)

Use one when it fits on a line and reads like a sentence. When it needs two
conditions and a nested loop, write the `for` loop — clever is not the goal.

### YOUR TURN 5

Using a **single comprehension**, build `deep_well_ids`: the IDs of wells whose
water level is deeper than 40 m.

Hint: `zip(well_ids, depths_to_water)` inside a comprehension works the same way
it does in a `for` loop.

In [ ]:
# YOUR TURN
deep_well_ids = ...

In [ ]:
# CHECK
assert deep_well_ids == ["B-07", "C-22", "E-11"], f"got {deep_well_ids}"
assert len(deep_well_ids) == 3
print(f"Wells deeper than 40 m: {deep_well_ids}. Correct.")

---

## Part 6 — Putting it together

Everything above, applied once. Read this cell rather than writing it — it is
the shape of code you will be writing by Week 6.

In [ ]:
import math


def well_report(well_id, depth_to_water, screen_bottom, land_surface_elev):
    """Summarize one well's condition as a dictionary."""
    is_dry = depth_to_water > screen_bottom
    remaining = screen_bottom - depth_to_water
    return {
        "well": well_id,
        "water_elev_m": round(land_surface_elev - depth_to_water, 2),
        "dry": is_dry,
        "screen_remaining_m": round(remaining, 2) if not is_dry else 0.0,
    }


land_elevs = [730.0, 742.5, 715.0, 760.2, 728.8]

reports = [
    well_report(w, d, s, e)
    for w, d, s, e in zip(well_ids, depths_to_water, screen_bottoms, land_elevs)
]

for r in reports:
    flag = "DRY" if r["dry"] else "   "
    print(f"{flag} {r['well']:5s} water at {r['water_elev_m']:7.2f} m elevation, "
          f"{r['screen_remaining_m']:5.1f} m of screen left")

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks — the order you happened to run things in is not
   the order the notebook is in
3. Save

## What's due

**HW 1 — My first Jupyter notebook**, Wednesday 9/9 at 11:59pm, through D2L.
Submit the `.ipynb` **with outputs kept**, so I can see that it ran.

## Next week

`numpy` arrays, and using them to solve equations you can't solve by hand:
bisection, and stepping a reservoir forward in time.

## Stuck?

- Read the **last line** of the traceback first. It says what went wrong. The
  lines above say where.
- `SyntaxError` on a line that looks fine usually means the problem is on the
  line *above* — an unclosed bracket or a missing `:`.
- `IndentationError` means Python can't tell where a block starts or stops.
  Four spaces, always.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.